# SINAN data engineering

This notebook cleans and prepares `sinan-db.csv` using the same feature logic described in `features.md` for the Recife dataset.


## Pipeline

- remove `DS_OBS`
- drop rows with no symptom/risk values
- drop columns with `>=28%` missingness except protected core fields
- translate Portuguese feature names/values
- convert age code to years
- create timeline, seasonality, composite scores, and interactions
- produce `sinan_single_type.csv`


In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
SINAN_PATH = Path(os.getenv("SINAN_PATH"))
OUTPUT_PATH = Path('sinan_fin.csv')
MISSING_THRESHOLD = 0.28

RAW_SYMPTOM_RISK = [
    "FEBRE","MIALGIA","CEFALEIA","EXANTEMA","VOMITO","NAUSEA","DOR_COSTAS","CONJUNTVIT",
    "ARTRITE","ARTRALGIA","PETEQUIA_N","LEUCOPENIA","LACO","DOR_RETRO","DIABETES",
    "HEMATOLOG","HEPATOPAT","RENAL","HIPERTENSA","ACIDO_PEPT","AUTO_IMUNE"
]

RENAME = {
    "DT_NOTIFIC":"notification_date",
    "SEM_NOT":"notification_epiweek",
    "NU_ANO":"notification_year",
    "DT_SIN_PRI":"first_symptom_date",
    "SEM_PRI":"first_symptom_epiweek",
    "DT_INVEST":"investigation_date",
    "NU_IDADE_N":"age_code",
    "CS_SEXO":"sex",
    "CS_GESTANT":"pregnancy_status",
    "CS_RACA":"race",
    "FEBRE":"fever",
    "MIALGIA":"myalgia",
    "CEFALEIA":"headache",
    "EXANTEMA":"rash",
    "VOMITO":"vomiting",
    "NAUSEA":"nausea",
    "DOR_COSTAS":"back_pain",
    "CONJUNTVIT":"conjunctivitis",
    "ARTRITE":"arthritis",
    "ARTRALGIA":"arthralgia",
    "PETEQUIA_N":"petechiae",
    "LEUCOPENIA":"leukopenia",
    "LACO":"tourniquet_test",
    "DOR_RETRO":"retroorbital",
    "DIABETES":"diabetes",
    "HEMATOLOG":"hematologic_disease",
    "HEPATOPAT":"liver_disease",
    "RENAL":"kidney_disease",
    "HIPERTENSA":"hypertension",
    "ACIDO_PEPT":"peptic_acid_disease",
    "AUTO_IMUNE":"autoimmune_disease",
    "CLASSI_FIN":"final_classification_code",
    "CRITERIO":"confirmation_criterion_code",
}

SYMPTOM_FEATURES = [
    "fever","headache","myalgia","rash","nausea","vomiting","back_pain","conjunctivitis",
    "arthritis","arthralgia","petechiae","retroorbital"
]

RISK_FEATURES = [
    "diabetes","hypertension","hematologic_disease","kidney_disease","liver_disease",
    "peptic_acid_disease","autoimmune_disease"
]

BI_FEATURES = SYMPTOM_FEATURES + RISK_FEATURES

YES_NO_MAP = {
    1:1, 1.0:1, "1":1, "1.0":1,
    2:0, 2.0:0, "2":0, "2.0":0,
    9:np.nan, 9.0:np.nan, "9":np.nan, "9.0":np.nan
}

# Helper functions
## normalize func:
## boolean agreament:
Convert to 1 and 0
## Age converter
Age is decoded in hours, days, months and years in the original dataset -> transform to year.

## Clinical time-window categories
Sort them by days
- `very early`: 0-1
- `early acute`: 2-3
- `mid-acute`: 4-5
- `late acute first-week illness`: 6-7
- `subacute / persistent symptoms`: 8-14
- `prolonged illness`: 15-21
- `very prolonged illness`: 22+
#### based on dengue clinical-phase timing from WHO/CDC-style clinical guidance

## Targets with severity (for future expansion if possible)
Vision: know Dengue or not Dengue, if Dengue then goes deeper to severity

In [44]:
def clean_code(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    return s[:-2] if s.endswith(".0") else s

def binary_value(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    return YES_NO_MAP.get(x, YES_NO_MAP.get(s, np.nan))

def age_code_to_years(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    m = re.match(r"^(\d)(\d+)$", s)
    if not m:
        return np.nan
    unit = int(m.group(1))
    value = int(m.group(2))
    if unit == 1:
        return value / (24 * 365.25)
    if unit == 2:
        return value / 365.25
    if unit == 3:
        return value / 12
    if unit == 4:
        return float(value)
    return np.nan

def bin_days(series):
    s = pd.to_numeric(series, errors="coerce")
    out = pd.Series(pd.NA, index=s.index, dtype="object")
    out[s < 0] = "negative"
    out[(s >= 0) & (s <= 1)] = "0-1"
    out[(s >= 2) & (s <= 3)] = "2-3"
    out[(s >= 4) & (s <= 5)] = "4-5"
    out[(s >= 6) & (s <= 7)] = "6-7"
    out[(s >= 8) & (s <= 14)] = "8-14"
    out[(s >= 15) & (s <= 21)] = "15-21"
    out[s >= 22] = "22+"
    return out

def safe_col(df, col):
    if col in df.columns:
        return df[col].fillna(0)
    return pd.Series(0, index=df.index)

# Main processing pipeline
## Clean
- Any symptoms or risk NaN in any row -> removed.
- Removed columns with more than 28% missing.
* Note: for pandas, `int64` cannot handle NaN values. If a col has NaN, pd will "upcast" the col to `float64` -> ❌; `Int64` store both int and pd.NA without converting the data to floats.
## Feature Engineering
- Phase windows
- Bin age group (infant, young child, and elderly).
- Rainy season (March to August) **Based on Recife, Pernambuco, Brazil** -> later must take account into VN seasons.
## Preprocess
- Fill any Risk Features NaN with 0.

In [45]:
df = pd.read_csv(SINAN_PATH, low_memory=False)
raw_rows, raw_cols = df.shape
print("Raw shape:", df.shape)

df = df.replace(r"^\s*$", pd.NA, regex=True)

for c in ["Unnamed: 0", "index", "DS_OBS"]:
    if c in df.columns:
        df = df.drop(columns=[c])

existing_filter_cols = [c for c in RAW_SYMPTOM_RISK if c in df.columns]
rows_before = len(df)
df = df.dropna(subset=existing_filter_cols, how="all").copy()
rows_removed = rows_before - len(df)

missing_before_drop = df.isna().mean().sort_values(ascending=False)
protected = {
    "CLASSI_FIN", "CRITERIO", "DT_SIN_PRI", "DT_NOTIFIC", "DT_INVEST",
    "NU_IDADE_N", "CS_SEXO", "CS_GESTANT", "CS_RACA"
}
cols_to_drop = [
    c for c in missing_before_drop[missing_before_drop >= MISSING_THRESHOLD].index.tolist()
    if c not in protected
]
df = df.drop(columns=cols_to_drop).copy()
df = df.rename(columns={k:v for k, v in RENAME.items() if k in df.columns})

# Binary conversion
for col in BI_FEATURES:
    if col in df.columns:
        df[col] = df[col].map(binary_value).astype("Float64")

# Categorical translation
if "sex" in df.columns:
    df["sex"] = df["sex"].astype("string").map({"M":"Male", "F":"Female", "I":"Unknown"})

if "pregnancy_status" in df.columns:
    df["pregnancy_status"] = df["pregnancy_status"].astype("string").map({
        "1":"1st_trimester", "1.0":"1st_trimester",
        "2":"2nd_trimester", "2.0":"2nd_trimester",
        "3":"3rd_trimester", "3.0":"3rd_trimester",
        "4":"Unknown", "4.0":"Unknown",
        "5":"Not_pregnant", "5.0":"Not_pregnant",
        "6":"Not_applicable", "6.0":"Not_applicable",
        "9":"Unknown", "9.0":"Unknown",
    })

if "race" in df.columns:
    df["race"] = df["race"].astype("string").map({
        "1":"White", "1.0":"White",
        "2":"Black", "2.0":"Black",
        "3":"Asian", "3.0":"Asian",
        "4":"Brown", "4.0":"Brown",
        "5":"Indigenous", "5.0":"Indigenous",
        "9":"Unknown", "9.0":"Unknown",
    })

# Age and demographic interactions
df["age"] = df["age_code"].apply(age_code_to_years) if "age_code" in df.columns else np.nan
df["is_pregnant"] = df.get("pregnancy_status", pd.Series(pd.NA, index=df.index)).isin([
    "1st_trimester", "2nd_trimester", "3rd_trimester"
]).astype(int)
df["female_x_pregnancy"] = ((df.get("sex") == "Female") & (df["is_pregnant"] == 1)).astype(int)
df["high_risk_age"] = ((df["age"] < 5) | (df["age"] >= 60)).astype(int)

# Date features
for col in ["first_symptom_date", "notification_date", "investigation_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)

df["sign_to_note"] = (df["notification_date"] - df["first_symptom_date"]).dt.days
df["sign_to_proper_care"] = (df["investigation_date"] - df["first_symptom_date"]).dt.days
df["note_to_proper_care"] = df["sign_to_proper_care"] - df["sign_to_note"]

df["note_bin"] = bin_days(df["sign_to_note"])
df["proper_care_bin"] = bin_days(df["sign_to_proper_care"])

df["early_contact"] = df["sign_to_note"].between(0, 3, inclusive="both").astype(int)
df["acute_window"] = df["sign_to_note"].between(0, 7, inclusive="both").astype(int)
df["warning_window"] = df["sign_to_note"].between(3, 7, inclusive="both").astype(int)
df["subacute_window"] = df["sign_to_note"].between(8, 14, inclusive="both").astype(int)
df["persistent_over_14d"] = (df["sign_to_note"] > 14).astype(int)

# Seasonality
df["onset_month"] = df["first_symptom_date"].dt.month
df["onset_dayofyear"] = df["first_symptom_date"].dt.dayofyear
df["onset_quarter"] = df["first_symptom_date"].dt.quarter
df["onset_weekofyear"] = df["first_symptom_date"].dt.isocalendar().week.astype("Float64")

df["in_rainy_season"] = df["onset_month"].isin([3,4,5,6,7,8]).astype(int)

df["onset_month_sin"] = np.sin(2 * np.pi * df["onset_month"] / 12)
df["onset_month_cos"] = np.cos(2 * np.pi * df["onset_month"] / 12)
df["onset_dayofyear_sin"] = np.sin(2 * np.pi * df["onset_dayofyear"] / 365.25)
df["onset_dayofyear_cos"] = np.cos(2 * np.pi * df["onset_dayofyear"] / 365.25)
df["onset_quarter_sin"] = np.sin(2 * np.pi * df["onset_quarter"] / 4)
df["onset_quarter_cos"] = np.cos(2 * np.pi * df["onset_quarter"] / 4)
df["onset_weekofyear_sin"] = np.sin(2 * np.pi * df["onset_weekofyear"] / 52)
df["onset_weekofyear_cos"] = np.cos(2 * np.pi * df["onset_weekofyear"] / 52)

# Composite features
df["metabolic_comorbidity_score"] = safe_col(df, "diabetes") + safe_col(df, "hypertension")

df["immunocompromised_flag"] = (
    (safe_col(df, "autoimmune_disease") == 1) |
    (safe_col(df, "hematologic_disease") == 1) |
    (safe_col(df, "kidney_disease") == 1) |
    (safe_col(df, "liver_disease") == 1)
).astype(int)

df["joint_score"] = (
    safe_col(df, "myalgia")
    + safe_col(df, "back_pain")
    + safe_col(df, "arthritis")
    + safe_col(df, "arthralgia")
)

df["gi_score"] = (
    safe_col(df, "nausea")
    + 2 * safe_col(df, "vomiting")
    + 0.75 * safe_col(df, "peptic_acid_disease")
    + 0.25 * (safe_col(df, "kidney_disease") + safe_col(df, "liver_disease"))
)

df["dengue_triad"] = (
    safe_col(df, "fever")
    + safe_col(df, "headache")
    + safe_col(df, "myalgia")
    + safe_col(df, "retroorbital")
)

df["dengue_warning_triad"] = (
    safe_col(df, "warning_window")
    + safe_col(df, "gi_score")
    + safe_col(df, "petechiae")
    + safe_col(df, "retroorbital")
)

df["chik_triad"] = (
    safe_col(df, "fever")
    + 2 * pd.concat([safe_col(df, "rash"), safe_col(df, "conjunctivitis")], axis=1).max(axis=1)
    + pd.concat([safe_col(df, "arthralgia"), safe_col(df, "arthritis")], axis=1).max(axis=1)
    + 0.5 * safe_col(df, "myalgia")
)

burden_cols = SYMPTOM_FEATURES + RISK_FEATURES
df["total_symptom_burden"] = df[burden_cols].fillna(0).sum(axis=1)

df["joint_gi_ratio"] = df["joint_score"] / (df["gi_score"] + 1)
df["dengue_warning_time_x_petechiae"] = safe_col(df, "warning_window") * safe_col(df, "petechiae")
df["age_x_arthritis"] = safe_col(df, "age") * safe_col(df, "arthritis")

# Targets
df["final_classification_code"] = df["final_classification_code"].map(clean_code).astype("string")

df["target_3class"] = df["final_classification_code"].map({
    "1":"Other", "2":"Other", "5":"Other", "8":"Other",
    "10":"Dengue", "11":"Dengue", "12":"Dengue", "13":"Chikungunya"
})

df["target"] = df["final_classification_code"].map({
    "1":"Other", "2":"Other", "5":"Other", "8":"Other",
    "10":"Dengue_mild", "11":"Dengue_warning",
    "12":"Dengue_severe", "13":"Chikungunya"
})

Raw shape: (57445, 147)


In [49]:
#
FINAL_COLS = [
    "target_3class", "target",
    "age", "sex", "pregnancy_status", "race",
    "is_pregnant", "female_x_pregnancy", "high_risk_age",
    "fever", "headache", "myalgia", "rash", "nausea", "vomiting",
    "back_pain", "conjunctivitis", "arthritis", "arthralgia",
    "petechiae", "retroorbital",
    "diabetes", "hypertension", "hematologic_disease", "kidney_disease",
    "liver_disease", "peptic_acid_disease", "autoimmune_disease",
    "metabolic_comorbidity_score", "immunocompromised_flag",
    "sign_to_note", "sign_to_proper_care", "note_to_proper_care",
    "note_bin", "proper_care_bin", "early_contact", "acute_window",
    "warning_window", "subacute_window", "persistent_over_14d",
    "onset_month", "onset_dayofyear", "onset_quarter", "onset_weekofyear",
    "in_rainy_season", "onset_month_sin", "onset_month_cos",
    "onset_dayofyear_sin", "onset_dayofyear_cos",
    "onset_quarter_sin", "onset_quarter_cos",
    "onset_weekofyear_sin", "onset_weekofyear_cos",
    "joint_score", "gi_score", "dengue_triad", "dengue_warning_triad",
    "chik_triad", "total_symptom_burden", "joint_gi_ratio",
    "dengue_warning_time_x_petechiae", "age_x_arthritis",
]

FINAL_COLS = [c for c in FINAL_COLS if c in df.columns]
final_df = df[FINAL_COLS].copy()
final_df.reset_index(inplace=True)
final_df.dropna(subset=["target", "target_3class"], inplace=True)
final_df = final_df.drop(columns=["index"])

final_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Final shape:", final_df.shape)
print(final_df["target_3class"].value_counts(dropna=False))
print("Saved:", OUTPUT_PATH)

Final shape: (38327, 62)
target_3class
Other          27056
Dengue         11024
Chikungunya      247
Name: count, dtype: int64
Saved: sinan_fin.csv


In [50]:
final_df["target"].value_counts(dropna=False)

target
Other             27056
Dengue_mild       10936
Chikungunya         247
Dengue_warning       63
Dengue_severe        25
Name: count, dtype: int64

In [51]:
final_df.head()

,target_3class,target,age,sex,pregnancy_status,race,is_pregnant,female_x_pregnancy,high_risk_age,fever,...,onset_weekofyear_cos,joint_score,gi_score,dengue_triad,dengue_warning_triad,chik_triad,total_symptom_burden,joint_gi_ratio,dengue_warning_time_x_petechiae,age_x_arthritis
0,Other,Other,34.0,Female,Not_pregnant,Brown,0,0,0,1.0,...,-0.748511,1.0,0.0,3.0,0.0,1.5,3.0,1.0,0.0,0.0
1,Other,Other,29.0,Female,Not_pregnant,Brown,0,0,0,1.0,...,-0.992709,0.0,0.0,2.0,1.0,1.0,2.0,0.0,0.0,0.0
2,Dengue,Dengue_mild,54.0,Female,Unknown,White,0,0,0,0.0,...,0.970942,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Other,Other,18.0,Male,Not_applicable,Indigenous,0,0,0,1.0,...,-0.568065,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0
4,Other,Other,21.0,Female,Not_pregnant,Brown,0,0,0,1.0,...,-0.992709,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0
